# DINOv2 Multi-View v4 — Inference Only

纯推理 notebook：读取 best_model.pt + DINOv2 权重 → 推理 test 集 → submission.csv

**前置条件：**
1. DINOv2 权重已上传为 Kaggle Dataset (`rsna-dinov2-weights`，包含 `dinov2_vits14.pth`)
2. best_model.pt 已上传为 Kaggle Dataset (`rsna-knee-v4-best-model`，包含 `best_model.pt`)
3. 竞赛数据已挂载

**不训练，不访问网络。**


## 1. Setup


In [ ]:
# ============================================================
# Inference v4: Setup — 禁网，仅推理
# ============================================================
# 竞赛环境预装 timm / pydicom / opencv / sklearn，无需 pip install
print("Setup complete.")



## 2. Imports


In [ ]:
# ============================================================
# Imports
# ============================================================
from __future__ import annotations
import gc, math, os, re, sys, time
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import pydicom
import cv2
from sklearn.metrics import roc_auc_score

IS_MAIN = True
print("Imports OK.")



## 3. Configuration


In [ ]:
# ============================================================
# Configuration — 纯推理（不训练）
# ============================================================

TARGET_COLUMNS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]
N_CLASSES = len(TARGET_COLUMNS)

# ---- 6 Clinical Slots ----
SLOTS = [
    ("SAG_FLUID_FS",   "Sagittal", True,  True),
    ("COR_FLUID_FS",   "Coronal",  True,  True),
    ("AX_FLUID_FS",    "Axial",    True,  True),
    ("SAG_FLUID_NOFS", "Sagittal", True,  False),
    ("COR_T1",         "Coronal",  False, False),
    ("SAG_T1",         "Sagittal", False, False),
]
N_SLOT = len(SLOTS)

# ---- Anatomical Priors ----
SLOT_PRIORS = {
    "ACL": (0, 3, 5), "MCL": (1, 4),
    "Medial Meniscus": (0, 1, 3, 4), "Lateral Meniscus": (0, 1, 3, 4),
    "Medial OA": (1, 4, 5), "Lateral OA": (1, 4, 5),
    "PF OA": (0, 2, 5), "Effusion": (0, 2), "Synovitis": (0, 2),
    "Baker's": (0,), "Contusion": (0, 1, 2), "Fracture": (0, 1, 2, 4, 5),
}

# ---- Diagnostic-specific TTA pooling ----
DIAG_POOL = {
    "Fracture": "max", "Contusion": "max",
    "Medial Meniscus": "max", "Lateral Meniscus": "max",
    "Baker's": "max",
    "ACL": "top2", "MCL": "top2",
}

CFG = {
    # --- Paths (修改为你自己的 Kaggle Dataset 路径) ---
    "comp_input":      "/kaggle/input/competitions/rsna-knee-abnormality-detection",
    "dinov2_weights":  "/kaggle/input/datasets/easoncyy/rsna-dinov2-weights/dinov2_vits14.pth",
    "best_model":      "/kaggle/input/datasets/easoncyy/rsna-knee-v4-best-model/best_model.pt",
    "test_dicom_subdir": "test_series",
    "output_dir":      "/kaggle/working",

    # --- Data ---
    "image_size": 224,
    "crop_mm": 160.0,
    "cache_slices": 9,
    "group_size": 3,
    "center_pct": (0.2, 0.8),

    # --- Model (必须与训练时一致) ---
    "dinov2_variant": "vit_small_patch14_dinov2.lvd142m",
    "cls_dim": 384,
    "feature_dim": 1152,
    "slot_hidden": 256,
    "num_classes": 12,
    "unfreeze_layers": 6,
    "dropout": 0.2,

    # --- Inference ---
    "pix_threads": 4,
    "batch_size": 8,
}

# Device
N_GPUS = torch.cuda.device_count()
DEVICE = torch.device("cuda" if N_GPUS > 0 else "cpu")

if IS_MAIN:
    print(f"GPUs: {N_GPUS} | Device: {DEVICE}")
    print("--- Inference v4: Test Set Only ---")
    for k, v in CFG.items():
        print(f"  {k}: {v}")



## 4. Slot Matching + Test DICOM Scan


In [ ]:
# ============================================================
# Slot Matching + DICOM Header Scan (test set)
# ============================================================

_SEP = re.compile(r'[_\-.]')
_FATSAT_RX = re.compile(
    r'\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|'
    r'water excit|\btirm\b|\bsting\b|\bfatsup\b'
)
_T1_RX = re.compile(r'\bt1\b|\bt1w\b')
_T2_RX = re.compile(r'\bt2\b|\bt2w\b')
_PD_RX = re.compile(r'\bpd\b|\bpdw\b|proton|\bdp\b|dens')


def _find_dicom_files(series_dir):
    """列出 DICOM 文件（不依赖 .dcm 扩展名，test 集可能无后缀）。"""
    sd = Path(series_dir)
    if not sd.is_dir():
        return []
    all_files = sorted([f for f in sd.iterdir() if f.is_file()])
    dcm = [f for f in all_files if f.suffix == '.dcm']
    return dcm if dcm else [f for f in all_files if not f.name.startswith('.')]


def _scan_test_dicoms(dicom_root):
    """扫描 test DICOM 目录，从 header 推断 plane / fluid / fatsat。"""
    rows = []
    root = Path(dicom_root)
    if not root.exists():
        return rows
    for study_dir in sorted(root.iterdir()):
        if not study_dir.is_dir():
            continue
        study_uid = study_dir.name
        for series_dir in sorted(study_dir.iterdir()):
            if not series_dir.is_dir():
                continue
            series_uid = series_dir.name
            dcm_files = _find_dicom_files(series_dir)
            if not dcm_files:
                continue
            try:
                ds = pydicom.dcmread(str(dcm_files[0]), stop_before_pixels=True, force=True)

                # Anatomical Plane
                iop = getattr(ds, 'ImageOrientationPatient', None)
                plane = 'Axial'
                if iop is not None and len(iop) >= 6:
                    try:
                        row_cos = np.array([float(iop[0]), float(iop[1]), float(iop[2])])
                        col_cos = np.array([float(iop[3]), float(iop[4]), float(iop[5])])
                        normal = np.cross(row_cos, col_cos)
                        dominant = int(np.argmax(np.abs(normal)))
                        plane = {0: 'Sagittal', 1: 'Coronal', 2: 'Axial'}[dominant]
                    except Exception:
                        pass

                # Fat Suppression
                desc = str(getattr(ds, 'SeriesDescription', '')).lower()
                scan_opts = str(getattr(ds, 'ScanOptions', '')).upper()
                fs_kw = ['fs', 'fatsat', 'fat sat', 'stir', 'spair', 'spir', 'we',
                         'water excit', 'tirm', 'fatsup']
                has_fs = any(kw in desc for kw in fs_kw)
                has_fs = has_fs or any(kw in scan_opts for kw in ['FS', 'FATSAT', 'SPAIR', 'SPIR'])

                # Fluid Sensitive
                seq_name = str(getattr(ds, 'SequenceName', '')).lower()
                is_t1 = any(kw in desc or kw in seq_name for kw in ['t1', 't1w'])
                is_t2 = any(kw in desc or kw in seq_name for kw in ['t2', 't2w'])
                is_pd = any(kw in desc for kw in ['pd', 'pdw', 'proton', 'dp', 'dens'])
                has_fluid = (is_t2 or is_pd) and not is_t1

                rows.append({
                    'StudyInstanceUID': study_uid,
                    'SeriesInstanceUID': series_uid,
                    'Anatomical_Plane': plane,
                    'Fluid_Sensitive': 1 if has_fluid else 0,
                    'Fat_Suppression': 1 if has_fs else 0,
                })
            except Exception:
                continue
    return rows


# ---- Slot Matching ----
def match_slots_for_study(study_series_df):
    """为单个 study 的每个 slot 匹配最优 series。"""
    slots_found = {}
    for slot_name, plane, fluid, fatsat in SLOTS:
        candidates = study_series_df[
            (study_series_df['Anatomical_Plane'] == plane)
            & (study_series_df['Fluid_Sensitive'] == (1 if fluid else 0))
            & (study_series_df['Fat_Suppression'] == (1 if fatsat else 0))
        ]
        if len(candidates) == 0 and not fluid:
            candidates = study_series_df[
                (study_series_df['Anatomical_Plane'] == plane)
                & (study_series_df['Fluid_Sensitive'] == 0)
            ]
        if len(candidates) > 0:
            best = candidates.sort_values('n_slices', ascending=False).iloc[0]
            slots_found[slot_name] = {
                'series_uid': best['SeriesInstanceUID'],
                'dir': best['dir'],
                'n_slices': int(best['n_slices']),
                'plane': plane,
            }
        else:
            slots_found[slot_name] = None
    return slots_found


def build_study_slot_map(series_meta, dicom_root):
    """为所有 study 构建 slot→series 映射。"""
    df = series_meta.copy()
    df['StudyInstanceUID'] = df['StudyInstanceUID'].astype(str)
    df['SeriesInstanceUID'] = df['SeriesInstanceUID'].astype(str)

    dirs, n_slices_list = [], []
    for _, row in df.iterrows():
        d = str(dicom_root / row['StudyInstanceUID'] / row['SeriesInstanceUID'])
        dirs.append(d)
        if os.path.isdir(d):
            files = [f for f in os.listdir(d) if os.path.isfile(os.path.join(d, f))]
            n_dcm = len([f for f in files if f.endswith('.dcm')])
            if n_dcm == 0:
                n_dcm = len([f for f in files if not f.startswith('.')])
            n_slices_list.append(n_dcm)
        else:
            n_slices_list.append(0)
    df['dir'] = dirs
    df['n_slices'] = n_slices_list

    slot_map = {}
    for study_uid, grp in df.groupby('StudyInstanceUID'):
        slot_map[study_uid] = match_slots_for_study(grp)

    if IS_MAIN:
        slot_counts = {}
        for slots in slot_map.values():
            for name, sid in slots.items():
                slot_counts[name] = slot_counts.get(name, 0) + (1 if sid is not None else 0)
        n_studies = len(slot_map)
        print(f'Slot map: {n_studies} studies')
        for name, count in slot_counts.items():
            print(f'  {name:<18s}: {count:5d}/{n_studies} ({count/max(n_studies,1)*100:.0f}%)')

    return slot_map


print('Slot matching ready.')



## 5. DICOM I/O


In [ ]:
# ============================================================
# DICOM I/O — 空间排序 + 物理裁剪 + 侧性归一化
# ============================================================

PLANE_AXIS = {"Sagittal": 0, "Coronal": 1, "Axial": 2}


def _list_dcm_files(series_dir):
    """列出 DICOM 文件（不依赖 .dcm 扩展名）。"""
    sd = Path(series_dir)
    if not sd.is_dir():
        return []
    all_files = sorted(f.name for f in sd.iterdir() if f.is_file())
    dcm = [f for f in all_files if f.endswith('.dcm')]
    return dcm if dcm else [f for f in all_files if not f.startswith('.')]


def spatially_sorted_files(series_dir, plane=None):
    """按 ImagePositionPatient 空间排序。"""
    series_dir = Path(series_dir)
    files = _list_dcm_files(series_dir)
    if not files:
        return []

    axis = PLANE_AXIS.get(plane, 2)
    rows = []
    for fname in files:
        try:
            ds = pydicom.dcmread(
                str(series_dir / fname), stop_before_pixels=True, force=True,
                specific_tags=['ImagePositionPatient', 'InstanceNumber'])
            ipp = getattr(ds, 'ImagePositionPatient', None)
            instance = getattr(ds, 'InstanceNumber', None)
            if ipp is not None and len(ipp) >= 3:
                candidate = np.array(ipp[:3], dtype=np.float64)
                pos = float(candidate[axis]) if np.isfinite(candidate).all() else None
            else:
                pos = None
            inst_val = float(instance) if instance is not None else None
        except Exception:
            pos, inst_val = None, None
        rows.append((fname, pos, inst_val))

    positioned = [r for r in rows if r[1] is not None]
    threshold = max(2, int(0.8 * len(rows)))

    if len(positioned) >= threshold:
        rows.sort(key=lambda r: (
            r[1] if r[1] is not None else 0.0,
            r[2] if r[2] is not None else float('inf'),
        ))
    elif sum(r[2] is not None for r in rows) >= threshold:
        rows.sort(key=lambda r: (
            r[2] if r[2] is not None else float('inf'),
        ))

    return [r[0] for r in rows]


def normalise_laterality(image, plane, laterality):
    """右膝→左膝归一化。"""
    if laterality != 'R':
        return image
    if plane in ('Coronal', 'Axial'):
        return np.flip(image, axis=-1).copy()
    else:
        return np.flip(image, axis=0).copy()


def physical_crop(volume, px, crop_mm=160.0):
    """固定物理 FOV 裁剪。"""
    if px is None or not np.isfinite(px) or px <= 0:
        return volume
    desired = int(round(crop_mm / px))
    h, w = volume.shape[1], volume.shape[2]
    if not (16 < desired < min(h, w)):
        return volume
    cy, cx = h // 2, w // 2
    half = desired // 2
    return volume[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]


def read_series_volume(series_dir, plane=None, laterality=None,
                       image_size=224, crop_mm=160.0):
    """读取 DICOM → 排序 → 裁剪 → 侧性 → 归一化 → 缩放。"""
    sorted_files = spatially_sorted_files(series_dir, plane)
    if not sorted_files:
        return None, None

    series_dir = Path(series_dir)
    slices_info = []
    px = None

    for fname in sorted_files:
        try:
            ds = pydicom.dcmread(str(series_dir / fname), force=True)
            img = ds.pixel_array.astype(np.float32)
            slope = float(getattr(ds, 'RescaleSlope', 1) or 1)
            intercept = float(getattr(ds, 'RescaleIntercept', 0) or 0)
            img = img * slope + intercept
            if px is None:
                ps = getattr(ds, 'PixelSpacing', None)
                if ps is not None and len(ps) >= 1:
                    try:
                        px = float(ps[0])
                    except Exception:
                        pass
            slices_info.append(img)
        except Exception:
            slices_info.append(np.zeros((image_size, image_size), dtype=np.float32))

    if not slices_info:
        return None, None

    volume = np.stack(slices_info, axis=0)
    volume = physical_crop(volume, px, crop_mm)
    volume = normalise_laterality(volume, plane, laterality)

    v_low, v_high = np.percentile(volume, [1.0, 99.0])
    volume = np.clip(volume, v_low, v_high)
    denom = max(v_high - v_low, 1e-6)
    volume = (volume - v_low) / denom

    resized = []
    for img in volume:
        r = cv2.resize(img, (image_size, image_size), interpolation=cv2.INTER_LINEAR)
        resized.append(r)
    return np.stack(resized, axis=0).astype(np.float32), px


def sample_cache_slices(volume, n_cache=9, center_pct=(0.2, 0.8)):
    """从 volume central 60% 均匀采样 n_cache 切片。"""
    n_total = volume.shape[0]
    if n_total <= n_cache:
        indices = list(range(n_total))
        while len(indices) < n_cache:
            indices.append(indices[-1])
        return volume[np.array(indices)]

    low = int(center_pct[0] * (n_total - 1))
    high = int(center_pct[1] * (n_total - 1))
    if high <= low:
        low, high = 0, n_total - 1
    indices = np.unique(np.linspace(low, high, n_cache).astype(int))
    while len(indices) < n_cache:
        indices = np.append(indices, indices[-1])
    return volume[indices[:n_cache]]


print('DICOM I/O ready.')



## 6. Model Definition


In [ ]:
# ============================================================
# Model — SlotHead + MultiViewModel + 诊断池化
# ============================================================

class SlotHead(nn.Module):
    """Per-diagnosis attention over MRI slots with anatomical priors."""

    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden

        prior = torch.zeros(n_out, n_slot)
        for target_name, slot_indices in SLOT_PRIORS.items():
            if target_name in TARGET_COLUMNS:
                prior[TARGET_COLUMNS.index(target_name), list(slot_indices)] = 0.55
        self.register_buffer("slot_prior", prior)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        attention = (
            torch.einsum("bsh,oh->bos", h, self.query)
            / math.sqrt(self.hidden)
            + self.slot_prior.unsqueeze(0)
        )
        attention = attention.masked_fill(
            mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        context = self.drop(torch.einsum("bos,bsh->boh", attention, h))
        return (context * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias


class MultiViewModel(nn.Module):
    """DINOv2 + SlotHead for multi-view knee MRI."""

    def __init__(self, dinov2_model, n_slots=6, cls_dim=384,
                 n_classes=12, slot_hidden=256, dropout=0.2,
                 unfreeze_layers=6):
        super().__init__()
        self.n_slots = n_slots
        self.cls_dim = cls_dim
        self.feature_dim = cls_dim * 3
        self.unfreeze_layers = unfreeze_layers

        self.dinov2 = dinov2_model
        n_blocks = len(self.dinov2.blocks)
        if unfreeze_layers > 0:
            for p in self.dinov2.parameters():
                p.requires_grad = False
            unfreeze_start = max(0, n_blocks - unfreeze_layers)
            for block in self.dinov2.blocks[unfreeze_start:]:
                for p in block.parameters():
                    p.requires_grad = True
            if hasattr(self.dinov2, 'norm'):
                for p in self.dinov2.norm.parameters():
                    p.requires_grad = True

        self.head = SlotHead(
            dim=self.feature_dim, n_slot=n_slots, n_out=n_classes,
            hidden=slot_hidden, p=dropout)

        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def _extract_features(self, x_3ch):
        with torch.no_grad():
            features = self.dinov2.forward_features(x_3ch)
        cls = features[:, 0, :]
        patches = features[:, 1:, :]
        mean_p = patches.mean(dim=1)
        k = max(1, patches.shape[1] // 8)
        focal = patches.topk(k, dim=1).values.mean(dim=1)
        return torch.cat([cls, mean_p, focal], dim=1)

    def forward(self, images, mask):
        B, S = images.shape[:2]
        x = images.reshape(B * S, 3, images.shape[-2], images.shape[-1])
        x = x.float().div_(255.0)
        x = (x - self.mean) / self.std
        features = self._extract_features(x)
        features = features.reshape(B, S, -1)
        return self.head(features, mask)

    def train(self, mode=True):
        super().train(mode)
        self.dinov2.eval()
        return self


# ---- 诊断特异性 TTA 池化 ----
DIAG_POOL_IDX = {}
for target_name, mode in DIAG_POOL.items():
    if target_name in TARGET_COLUMNS:
        DIAG_POOL_IDX[TARGET_COLUMNS.index(target_name)] = mode


def diagnostic_pool(logits_windows, pool_idx=None):
    """对 [B, W, C] logits 应用诊断特异性池化。"""
    if pool_idx is None:
        pool_idx = DIAG_POOL_IDX

    B, W, C = logits_windows.shape
    probs = torch.sigmoid(logits_windows)
    result = probs.mean(dim=1)

    for j, mode in pool_idx.items():
        x = probs[:, :, j]
        if mode == 'max':
            result[:, j] = x.max(dim=1).values
        elif mode == 'top2':
            result[:, j] = x.topk(min(2, W), dim=1).values.mean(dim=1)

    return result


if IS_MAIN:
    print(f'Model ready. Diag pool targets: {list(DIAG_POOL_IDX.keys())}')



## 7. Build Test Cache


In [ ]:
# ============================================================
# Build Test Cache — 扫描 DICOM + 构建槽位映射 + 并行读取
# ============================================================

comp_input = Path(CFG['comp_input'])
test_dicom_root = comp_input / CFG['test_dicom_subdir']
output_dir = Path(CFG['output_dir'])

# ---- Load test.csv ----
test_df = pd.read_csv(comp_input / 'test.csv')
test_df['StudyInstanceUID'] = test_df['StudyInstanceUID'].astype(str)
print(f'Test studies (from test.csv): {len(test_df)}')

# ---- Build test series metadata ----
# 优先 test_series.csv，不足则扫描 DICOM headers
test_series_path = comp_input / 'test_series.csv'
test_series_loaded = False

# ★ 重写逻辑：CSV 结果不会被 DICOM scan 失败覆盖
test_slot_map = {}

if test_series_path.exists():
    test_series = pd.read_csv(test_series_path)
    test_series['StudyInstanceUID'] = test_series['StudyInstanceUID'].astype(str)
    test_series['SeriesInstanceUID'] = test_series['SeriesInstanceUID'].astype(str)
    print(f'test_series.csv: {len(test_series)} series, '
          f'{test_series["StudyInstanceUID"].nunique()} studies')

    test_slot_map = build_study_slot_map(test_series, test_dicom_root)
    csv_studies = len(test_slot_map)

    if csv_studies < max(10, len(test_df) * 0.5):
        print(f'CSV coverage ({csv_studies}/{len(test_df)}) insufficient, '
              f'scanning DICOM headers...')
        dicom_rows = _scan_test_dicoms(test_dicom_root)
        if dicom_rows:
            test_series = pd.DataFrame(dicom_rows)
            test_slot_map = build_study_slot_map(test_series, test_dicom_root)
            print(f'DICOM scan: {len(test_series)} series, '
                  f'{test_series["StudyInstanceUID"].nunique()} studies → '
                  f'{len(test_slot_map)} studies matched')
        else:
            # DICOM scan 失败 → 保留 CSV 结果（即使不完整）
            print(f'DICOM scan returned 0 rows, keeping CSV results ({csv_studies} studies)')
    # else: CSV 覆盖率够了，直接用
else:
    print('test_series.csv not found, scanning DICOM headers...')
    dicom_rows = _scan_test_dicoms(test_dicom_root)
    if dicom_rows:
        test_series = pd.DataFrame(dicom_rows)
        test_slot_map = build_study_slot_map(test_series, test_dicom_root)
        print(f'DICOM scan: {len(test_series)} series, '
              f'{len(test_slot_map)} studies matched')

test_studies = sorted(test_slot_map.keys())
print(f'Test studies with slot match: {len(test_studies)}/{len(test_df)}')

if len(test_studies) == 0:
    raise RuntimeError(
        'No test studies found with slot matching! '
        'Check that test DICOMs exist at: ' + str(test_dicom_root))

# ---- 快速侧性检测（每个 study 扫描一个 DICOM header）----
def _detect_laterality_fast(slot_map, dicom_root):
    laterality_map = {}
    root = Path(dicom_root)
    for study_uid, study_slots in slot_map.items():
        lat = None
        for slot_name, slot_info in study_slots.items():
            if slot_info is None:
                continue
            series_dir = Path(slot_info['dir']) if 'dir' in slot_info else None
            if series_dir is None or not series_dir.exists():
                continue
            dcm_files = _list_dcm_files(series_dir)
            if not dcm_files:
                continue
            try:
                ds = pydicom.dcmread(
                    str(series_dir / dcm_files[0]), stop_before_pixels=True, force=True,
                    specific_tags=['Laterality', 'ImageLaterality', 'ImagePositionPatient'])
                for tag_name in ['Laterality', 'ImageLaterality']:
                    val = getattr(ds, tag_name, None)
                    if val is not None:
                        val = str(val).strip().upper()
                        if val and val[0] in ('L', 'R'):
                            lat = val[0]
                            break
                if lat is not None:
                    break
                ipp = getattr(ds, 'ImagePositionPatient', None)
                if ipp is not None and len(ipp) >= 1:
                    try:
                        x = float(str(ipp[0]).split('\\')[0].split('|')[0])
                        if abs(x) >= 5.0:
                            lat = 'R' if x < 0 else 'L'
                            break
                    except Exception:
                        pass
            except Exception:
                continue
        laterality_map[study_uid] = lat
    return laterality_map


t_lat = time.time()
laterality_map = _detect_laterality_fast(test_slot_map, test_dicom_root)
n_lat = sum(1 for v in laterality_map.values() if v is not None)
if IS_MAIN:
    print(f'Laterality detected: {n_lat}/{len(laterality_map)} studies '
          f'({n_lat/max(len(laterality_map),1)*100:.1f}%), '
          f'({time.time()-t_lat:.1f}s)')

# ---- Pre-allocate cache ----
n_test = len(test_studies)
TEST_CACHE = np.zeros((n_test, N_SLOT, CFG['cache_slices'], CFG['image_size'], CFG['image_size']), dtype=np.uint8)
TEST_MASK = np.zeros((n_test, N_SLOT), dtype=np.float32)
test_study_idx = {}

for row_idx, study_uid in enumerate(test_studies):
    test_study_idx[study_uid] = row_idx

# ---- Parallel DICOM read ----
def _read_slot_job(args):
    row_idx, slot_idx, slot_name, plane, slot_info, laterality = args
    if slot_info is None:
        return row_idx, slot_idx, None
    series_dir = Path(slot_info['dir']) if 'dir' in slot_info else None
    if series_dir is None or not series_dir.exists():
        return row_idx, slot_idx, None
    try:
        volume, px = read_series_volume(
            str(series_dir), plane=plane, laterality=laterality,
            image_size=CFG['image_size'], crop_mm=CFG['crop_mm'])
        if volume is None or volume.shape[0] < 3:
            return row_idx, slot_idx, None
        sampled = sample_cache_slices(
            volume, n_cache=CFG['cache_slices'], center_pct=CFG['center_pct'])
        sampled_uint8 = (sampled * 255).clip(0, 255).round().astype(np.uint8)
        return row_idx, slot_idx, sampled_uint8
    except Exception:
        return row_idx, slot_idx, None


jobs = []
for row_idx, study_uid in enumerate(test_studies):
    study_slots = test_slot_map[study_uid]
    lat = laterality_map.get(study_uid)
    for slot_idx, (slot_name, plane, fluid, fatsat) in enumerate(SLOTS):
        slot_info = study_slots.get(slot_name)
        if slot_info is not None:
            jobs.append((row_idx, slot_idx, slot_name, plane, slot_info, lat))

print(f'Decoding {len(jobs)} test slot-series (parallel, {CFG["pix_threads"]} threads)...')

t_cache = time.time()
completed = 0
failed = 0
with ThreadPoolExecutor(max_workers=CFG['pix_threads']) as pool:
    for row_idx, slot_idx, result in pool.map(_read_slot_job, jobs):
        completed += 1
        if result is not None:
            TEST_CACHE[row_idx, slot_idx] = result
            TEST_MASK[row_idx, slot_idx] = 1.0
        else:
            failed += 1
        if completed % 2000 == 0:
            elapsed = time.time() - t_cache
            eta = (elapsed / completed) * (len(jobs) - completed) / 60
            print(f'  [{completed}/{len(jobs)}] {completed/len(jobs)*100:.0f}% | '
                  f'{elapsed:.0f}s | ~{eta:.0f}min left', flush=True)

cache_time = time.time() - t_cache
total_series = int(TEST_MASK.sum())
print(f'\nTest cache built: {n_test} studies, {total_series} series, '
      f'{TEST_CACHE.nbytes/1024**3:.1f} GB in {cache_time:.0f}s')
print(f'  Avg slots/study: {total_series/max(n_test,1):.1f} | Failed: {failed}')

gc.collect()



## 8. Load Model Weights


In [ ]:
# ============================================================
# Load Model — DINOv2 权重 + best_model.pt checkpoint
# ============================================================

print('Loading DINOv2 backbone...')

dinov2_backbone = timm.create_model(
    CFG['dinov2_variant'], pretrained=False, num_classes=0,
    img_size=CFG['image_size'],
)

# ★ 加载 DINOv2 预训练权重（含 pos_embed 插值）
weights_path = Path(CFG['dinov2_weights'])
if weights_path.exists():
    state_dict = torch.load(weights_path, map_location='cpu', weights_only=True)

    if 'pos_embed' in state_dict:
        pos_ckpt = state_dict['pos_embed']
        pos_model = dinov2_backbone.pos_embed.data
        if pos_ckpt.shape != pos_model.shape:
            cls_ckpt = pos_ckpt[:, :1, :]
            patch_ckpt = pos_ckpt[:, 1:, :]
            grid_ckpt = int(math.isqrt(patch_ckpt.shape[1]))
            grid_model = int(math.isqrt(pos_model.shape[1] - 1))
            patch_ckpt = patch_ckpt.reshape(1, grid_ckpt, grid_ckpt, -1).permute(0, 3, 1, 2)
            patch_interp = F.interpolate(
                patch_ckpt, size=(grid_model, grid_model), mode='bicubic',
                antialias=True)
            patch_interp = patch_interp.permute(0, 2, 3, 1).reshape(1, -1, pos_model.shape[-1])
            state_dict['pos_embed'] = torch.cat([cls_ckpt, patch_interp], dim=1)
            print(f'  pos_embed interpolated: [{grid_ckpt}x{grid_ckpt}] -> [{grid_model}x{grid_model}]')

    dinov2_backbone.load_state_dict(state_dict, strict=True)
    print(f'  DINOv2 weights loaded: {weights_path}')
else:
    raise FileNotFoundError(f'DINOv2 weights not found: {weights_path}')

# ---- Build model ----
model = MultiViewModel(
    dinov2_model=dinov2_backbone,
    n_slots=N_SLOT, cls_dim=CFG['cls_dim'],
    n_classes=CFG['num_classes'], slot_hidden=CFG['slot_hidden'],
    dropout=0.0, unfreeze_layers=CFG['unfreeze_layers'],
).to(DEVICE)
model.eval()

# ---- Load best checkpoint ----
best_ckpt_path = Path(CFG['best_model'])
if not best_ckpt_path.exists():
    raise FileNotFoundError(
        f'Best model not found: {best_ckpt_path}\n'
        'Upload best_model.pt as a Kaggle Dataset and update CFG["best_model"].')

print(f'Loading checkpoint: {best_ckpt_path}')
ckpt = torch.load(best_ckpt_path, map_location='cpu', weights_only=False)

state_dict = ckpt['model']
first_key = next(iter(state_dict))
if first_key.startswith('module.'):
    state_dict = {k.replace('module.', '', 1): v for k, v in state_dict.items()}

# ★ 优先使用 EMA 权重
if ckpt.get('ema') and ckpt['ema'].get('shadow'):
    for name in state_dict:
        ema_key = name
        if ema_key in ckpt['ema']['shadow']:
            state_dict[name] = ckpt['ema']['shadow'][ema_key]
    print('  Using EMA weights for inference')

model.load_state_dict(state_dict, strict=False)
model.eval()
print(f'  Model loaded: epoch={ckpt.get("epoch")}, AUC={ckpt.get("auc", 0):.4f}')

# Report GPU memory
if torch.cuda.is_available():
    print(f'  GPU memory: {torch.cuda.memory_allocated()/1024**3:.1f} GB')



## 9. Inference -> submission.csv


In [ ]:
# ============================================================
# Test Inference — 7-Window TTA + 诊断池化 -> submission.csv
# ============================================================

N_WINDOWS = CFG['cache_slices'] - CFG['group_size'] + 1  # 7
print(f'Running TTA inference: {n_test} studies, {N_WINDOWS}-window TTA + diag pool')

test_probs = np.zeros((n_test, N_CLASSES), dtype=np.float32)

@torch.no_grad()
def infer_test_batch(indices, model):
    windows_list, masks_list, empty_mask = [], [], []
    for idx in indices:
        windows_list.append(torch.stack([
            torch.from_numpy(TEST_CACHE[idx, :, w:w + CFG['group_size']].copy())
            for w in range(N_WINDOWS)
        ], dim=0))
        masks_list.append(torch.from_numpy(TEST_MASK[idx].copy()))
        empty_mask.append(TEST_MASK[idx].sum() == 0)

    windows_batch = torch.stack(windows_list)
    mask_batch = torch.stack(masks_list)

    B, W = windows_batch.shape[0], N_WINDOWS
    flat = windows_batch.reshape(B * W, *windows_batch.shape[2:]).to(DEVICE)
    flat_mask = mask_batch.unsqueeze(1).expand(B, W, -1).reshape(B * W, -1).to(DEVICE)
    logits = model(flat, flat_mask)
    probs = diagnostic_pool(logits.reshape(B, W, -1).cpu())

    for i, is_empty in enumerate(empty_mask):
        if is_empty:
            probs[i] = 0.5
    return probs


t_infer = time.time()
batch_size = CFG['batch_size']
for start in range(0, n_test, batch_size):
    idx = list(range(start, min(start + batch_size, n_test)))
    test_probs[idx] = infer_test_batch(idx, model).numpy()
    if start % 200 == 0:
        print(f'  [{start}/{n_test}] {time.time()-t_infer:.0f}s', flush=True)

print(f'Test inference done in {time.time()-t_infer:.0f}s')

# ---- Build submission.csv ----
submission_rows = []
for row_idx, study_uid in enumerate(test_studies):
    row = {'StudyInstanceUID': study_uid}
    for j, c in enumerate(TARGET_COLUMNS):
        row[c] = float(test_probs[row_idx, j])
    submission_rows.append(row)

submission_df = pd.DataFrame(submission_rows)

# 确保所有 test.csv 中的 study 都在 submission 中
full_submission = test_df[['StudyInstanceUID']].merge(
    submission_df, on='StudyInstanceUID', how='left')
for c in TARGET_COLUMNS:
    full_submission[c] = full_submission[c].fillna(0.5)

submission_path = output_dir / 'submission.csv'
full_submission.to_csv(submission_path, index=False)
print(f'\nSubmission saved: {submission_path}')
print(f'  Studies: {len(full_submission)} (expected: {len(test_df)})')
print(f'  Mean prob: {full_submission[TARGET_COLUMNS].values.mean():.4f}')
print()

# Per-class stats
for c in TARGET_COLUMNS:
    vals = full_submission[c].values
    print(f'  {c:<20s}: mean={vals.mean():.4f}, std={vals.std():.4f}, '
          f'>0.5={np.mean(vals>0.5):.1%}')

print(f'\nDone! Submit {submission_path} to Kaggle.')

